# Day 30 — 단변량 군 비교: 분포, 효과크기, 신뢰구간

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-30.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-30.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 참가자 단위로 요약하기

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

# 한 사람 = 한 행이 되도록 반복 측정을 평균으로 요약
person = df.groupby([ID_COL, GROUP_COL], as_index=False)[FEATURES].mean()

print("요약 전 행 수:", len(df), "→ 요약 후 행 수:", len(person))
print()
print(person.groupby(GROUP_COL)[FEATURES].mean().round(2).T)

### 예시 2 — feature 하나: Cohen's d와 bootstrap 신뢰구간

예시 1에 이어서 실행합니다. 함수 두 개를 만들어 두면 내일도, 보고서를 쓸 때도 다시 씁니다.

In [ ]:
def cohens_d(a, b):
    """a - b 방향의 Cohen's d (합동 표준편차 사용)."""
    a, b = np.asarray(a), np.asarray(b)
    pooled_var = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(pooled_var)

def bootstrap_ci(a, b, stat, n_boot=5000, seed=0):
    """참가자를 복원추출해 stat(a, b)의 95% 신뢰구간을 구한다."""
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    boots = [stat(rng.choice(a, size=len(a), replace=True),
                  rng.choice(b, size=len(b), replace=True)) for _ in range(n_boot)]
    return np.percentile(boots, [2.5, 97.5])

feature = "tremor_band_power"
pd_vals = person.loc[person[GROUP_COL] == "PD", feature]
hc_vals = person.loc[person[GROUP_COL] == "HC", feature]

mean_diff = lambda a, b: a.mean() - b.mean()
lo, hi = bootstrap_ci(pd_vals, hc_vals, mean_diff)
d_lo, d_hi = bootstrap_ci(pd_vals, hc_vals, cohens_d)

print(f"{feature}  (PD n={len(pd_vals)}, HC n={len(hc_vals)})")
print(f"  평균 차이(PD - HC) = {mean_diff(pd_vals, hc_vals):.2f},  95% CI [{lo:.2f}, {hi:.2f}]")
print(f"  Cohen's d          = {cohens_d(pd_vals, hc_vals):.2f},  95% CI [{d_lo:.2f}, {d_hi:.2f}]")

### 예시 3 — 모든 feature: 효과크기 표, FDR, 그림

예시 2에 이어서 실행합니다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats

rows = []
for f in FEATURES:
    a = person.loc[person[GROUP_COL] == "PD", f]
    b = person.loc[person[GROUP_COL] == "HC", f]
    lo, hi = bootstrap_ci(a, b, cohens_d)
    p = stats.ttest_ind(a, b, equal_var=False).pvalue        # Welch t-검정
    rows.append({"feature": f, "mean_diff": a.mean() - b.mean(),
                 "cohens_d": cohens_d(a, b), "d_ci_low": lo, "d_ci_high": hi, "p": p})

table = pd.DataFrame(rows)
table["q_fdr"] = stats.false_discovery_control(table["p"])   # Benjamini-Hochberg
table = table.sort_values("cohens_d", key=abs, ascending=False)
print(table.round(3).to_string(index=False))

Path("reports").mkdir(exist_ok=True)
table.to_csv("reports/day30_effect_sizes.csv", index=False)

# 그림: 참가자 한 명 = 점 하나, 그 위에 boxplot
Path("figures").mkdir(exist_ok=True)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
for ax, f in zip(axes.ravel(), table["feature"]):
    groups = ["HC", "PD"]
    data = [person.loc[person[GROUP_COL] == g, f] for g in groups]
    ax.boxplot(data, positions=[0, 1], widths=0.5, showfliers=False)
    for x, vals in enumerate(data):
        ax.scatter(x + rng.uniform(-0.12, 0.12, len(vals)), vals, s=18, alpha=0.6)
    d = table.set_index("feature").loc[f]
    ax.set_title(f"{f}\nd = {d.cohens_d:.2f} [{d.d_ci_low:.2f}, {d.d_ci_high:.2f}]", fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f"{g} (n={len(v)})" for g, v in zip(groups, data)])
    ax.set_ylabel("participant mean (a.u.)")
fig.suptitle("Synthetic demo data: participant-level group comparison (d = PD - HC, bootstrap 95% CI)")
fig.tight_layout()
fig.savefig("figures/day30_group_comparison.png", dpi=150)
plt.show()
print("저장: figures/day30_group_comparison.png, reports/day30_effect_sizes.csv")